# HydroSeismic | Final Interactive Demo

Explore the combined USGS and GDACS incident dataset.
Priority is an experimental heuristic, **not an official warning or validated forecast**.


In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

BASE = Path("/home/ibm-user/hydroseismic_data")
DATA = BASE / "processed" / "incidents_master_final_scored.csv"
df = pd.read_csv(DATA)

print(f"Loaded {len(df)} incidents")
print("Columns:", ", ".join(df.columns))
display(df.head())


Loaded 288 incidents
Columns: incident_id, source, event_type, event_name, country, latitude, longitude, event_time_utc, magnitude, depth_km, official_alert_level, source_url, source_event_id, episode_id, priority_score, priority_band, magnitude_points, alert_points


,incident_id,source,event_type,event_name,country,latitude,longitude,event_time_utc,magnitude,depth_km,official_alert_level,source_url,source_event_id,episode_id,priority_score,priority_band,magnitude_points,alert_points
0,USGS-aka2026uopnnb,USGS,earthquake,"63 km ENE of Pedro Bay, Alaska",NaN,60.056000,-153.113000,2026-10-17T10:35:46.934000+00:00,0.60,5.000000,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,USGS-aka2026uopnnb,NaN,0.5,Low,0.514286,0.0
1,USGS-hv75059611,USGS,earthquake,"12 km NE of Pāhala, Hawaii",NaN,19.278833,-155.395828,2026-10-17T10:23:57.610000+00:00,1.97,30.260000,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,USGS-hv75059611,NaN,5.5,Low,5.544143,0.0
2,USGS-aka2026uoopfk,USGS,earthquake,"47 km N of Petersville, Alaska",NaN,62.924000,-150.815000,2026-10-17T10:07:45.535000+00:00,2.50,111.400000,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,USGS-aka2026uoopfk,NaN,8.9,Low,8.928571,0.0
3,USGS-hv75059601,USGS,earthquake,"12 km NE of Pāhala, Hawaii",NaN,19.286167,-155.399338,2026-10-17T10:06:59.710000+00:00,2.44,28.780001,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,USGS-hv75059601,NaN,8.5,Low,8.505143,0.0
4,USGS-ci41348511,USGS,earthquake,"4 km SSW of Redlands, CA",NaN,34.020000,-117.191167,2026-10-17T09:57:23.030000+00:00,0.74,5.120000,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,USGS-ci41348511,NaN,0.8,Low,0.782286,0.0


In [2]:
# Detect the relevant columns from the existing dataset
def find_col(candidates):
    for candidate in candidates:
        for col in df.columns:
            if col.lower().replace(" ", "_") == candidate:
                return col
    for candidate in candidates:
        for col in df.columns:
            if candidate in col.lower().replace(" ", "_"):
                return col
    return None

source_col = find_col(["source"])
type_col = find_col(["event_type", "event"])
priority_col = find_col(["priority_band", "priority", "priority_level"])
lat_col = find_col(["latitude", "lat"])
lon_col = find_col(["longitude", "lon", "lng"])

print("Detected columns:")
print("Source:", source_col)
print("Event type:", type_col)
print("Priority:", priority_col)
print("Latitude:", lat_col, "| Longitude:", lon_col)

if source_col is None or type_col is None or priority_col is None:
    raise ValueError("Could not identify source, event type, and priority columns. Inspect the column list above.")


Detected columns:
Source: source
Event type: event_type
Priority: priority_band
Latitude: latitude | Longitude: longitude


In [3]:
source_filter = widgets.Dropdown(
    options=["All"] + sorted(df[source_col].dropna().astype(str).unique().tolist()),
    value="All", description="Source:"
)
type_filter = widgets.Dropdown(
    options=["All"] + sorted(df[type_col].dropna().astype(str).unique().tolist()),
    value="All", description="Event:"
)
priority_filter = widgets.Dropdown(
    options=["All"] + sorted(df[priority_col].dropna().astype(str).unique().tolist()),
    value="All", description="Priority:"
)
out = widgets.Output()

def render_demo(*args):
    with out:
        clear_output(wait=True)
        view = df.copy()
        if source_filter.value != "All":
            view = view[view[source_col].astype(str) == source_filter.value]
        if type_filter.value != "All":
            view = view[view[type_col].astype(str) == type_filter.value]
        if priority_filter.value != "All":
            view = view[view[priority_col].astype(str) == priority_filter.value]

        print(f"Filtered incidents: {len(view)}")
        display(view.head(20))

        if lat_col and lon_col:
            coords = view[[lon_col, lat_col]].apply(pd.to_numeric, errors="coerce").dropna()
            if not coords.empty:
                plt.figure(figsize=(10, 5))
                plt.scatter(coords[lon_col], coords[lat_col], alpha=0.65)
                plt.xlabel("Longitude")
                plt.ylabel("Latitude")
                plt.title("HydroSeismic incident locations")
                plt.grid(True, alpha=0.25)
                plt.show()
            else:
                print("No valid coordinates in the selected records.")

        counts = view[priority_col].astype(str).value_counts()
        if not counts.empty:
            counts.plot(kind="bar", figsize=(7, 3), title="Priority distribution")
            plt.ylabel("Incident count")
            plt.tight_layout()
            plt.show()

        print("Data-quality checks")
        print("Duplicate rows:", int(view.duplicated().sum()))
        print("Missing values:", int(view.isna().sum().sum()))
        if lat_col and lon_col:
            print("Missing coordinates:", int(view[[lat_col, lon_col]].isna().any(axis=1).sum()))

for control in (source_filter, type_filter, priority_filter):
    control.observe(render_demo, names="value")

display(widgets.VBox([source_filter, type_filter, priority_filter, out]))
render_demo()


## Interpretation and safety

- Priority bands are heuristic classifications, not validated risk estimates.
- This demo does not predict future earthquakes or floods.
- It does not issue public warnings or trigger emergency actions.
- The view uses the saved master dataset. Refresh it separately before a presentation if you need the latest events.
